# 📊 05｜連鎖書店季報：多表合併，先核對對不上的資料

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/05_bookstore_merge/lesson.ipynb)

> 💡 **學習模式說明**：
> 本筆記本專為**完全不寫程式的初學者**設計，全程搭配 **Google Colab 內建的 Gemini AI** 進行協作。
> 你不需要手寫程式碼：先用 Prompt 指揮 AI，再對照下方已附的參考程式，執行並驗收數字與圖表。
> 🆕 **本例只新增一個重點**：三個月的訂單要**上下接起來**，再**左右對應**門市與商品資料；合併後一定要清點「對不上」的紀錄，並說明它們如何影響營收。

## 🪜 本例路線：先完成一次，再逐步加深
1. **必做主線（單元一～三）**：接起三個月訂單、統一門市代碼、對應門市與商品，清點對不上的資料，最後用**一張垂直長條圖**回答「第一季哪家門市營收最高？」。單元三後先停下來驗收。
2. **報表應用（單元四）**：匯出門市營收與「未匹配清單」，讓主管知道哪些營收還無法計算。
3. **選做挑戰（單元五～六）**：比較各門市三個月的營收變化，或看各門市的書籍類別組成。

---

## 🧭 資料專案介紹：貫徹「看懂資料與選對圖表」思維 SOP

資料：`data/orders_2025_01.csv`～`orders_2025_03.csv`（每月訂單）、`data/stores.csv`（門市主檔）、`data/products.csv`（商品主檔）。請先閱讀[本例資料卡](./data/DATA_CARD.md)。**訂單檔的一列代表：一筆訂單（一種書、一個數量）**；門市主檔一列是一家門市，商品主檔一列是一種書。

### 1. 欄位角色貼標籤
- ⚪ **標識 (ID)**：`訂單編號`（只能計數）；`門市代碼`、`商品編號` 是**合併用的鑰匙**，寫法必須一致才對得上。
- 🔵 **條件 (Dimension)**：`日期`（時間）；`門市`、`縣市`、`書名`、`類別`（類別，要從主檔合併過來）。
- 🟢 **指標 (Metric)**：`數量`（本，三月欄名叫「購買數量」）、`售價`（元，在商品主檔）➔ 營收＝數量 × 售價。

### 2. 五大提問積木與核心問題
- 🎯 **比較（必做）**：第一季哪一家門市的營收最高？
- 📅 **比較＋時間（選做）**：各門市 1、2、3 月的營收變化一致嗎？
- 🧩 **組成（選做）**：各門市的營收，由哪些書籍類別組成？

### 3. 邊界反向檢核防呆（不可觸犯的紅線）
- ⚠️ **欄名要先統一**：三月的「購買數量」不改名就上下合併，會變成兩個各缺一部分的欄位。
- ⚠️ **鑰匙寫法要一致**：「s02 」和「S02」對電腦來說是不同的值，要先去空白、轉大寫。
- ⚠️ **合併後清點未匹配**：訂單有門市 S07，但門市主檔沒有；有商品 P013，但商品主檔沒有。用 inner join 會**默默刪掉**這些訂單，營收就少算了。
- ⚠️ **無法計價就說無法計價**：P013 沒有售價，不能猜價格補上；營收要註明「不含無法計價的訂單」，並列出影響筆數與本數。
- ⚠️ **沒有訂單 ≠ 營收很差**：S06 高雄店在主檔中卻沒有任何訂單，可能是資料缺漏，不能直接寫成「高雄店營收為 0」。

### 4. 決定最適圖表
- **6 個門市比營收** ➔ **垂直長條圖**（類別少；Y 軸從 0 開始，單位「元」；S07 用灰色標示「門市資料缺」）。
- **各門市 3 個月營收（選做）** ➔ **分組長條圖**（只有 3 個時間點，用長條比用折線清楚）。
- **各門市的類別組成（選做）** ➔ **百分比堆疊長條圖**（每家門市都是 100%，看結構）。

---

## 🤖 Colab Gemini AI 操作方式
1. 每次只複製一個單元的 Prompt，貼到 Colab Gemini 對話框。
2. 比較 AI 的答覆與下方參考程式，執行目前單元的儲存格。
3. 先看懂輸出並回答驗收問題，再進入下一單元。

In [ ]:
# 第 0 步：下載範例資料並安裝繁體中文字型（請先點擊左側 ▶ 執行此儲存格）
import os
from pathlib import Path
from urllib.request import urlretrieve

# 1. 下載資料（已存在就不重複下載；若下載失敗，請手動上傳到 data/）
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
files = {
    'orders_2025_01.csv': 'https://raw.githubusercontent.com/roberthsu2003/PythonForDataAnalysis/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/05_bookstore_merge/data/orders_2025_01.csv',
    'orders_2025_02.csv': 'https://raw.githubusercontent.com/roberthsu2003/PythonForDataAnalysis/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/05_bookstore_merge/data/orders_2025_02.csv',
    'orders_2025_03.csv': 'https://raw.githubusercontent.com/roberthsu2003/PythonForDataAnalysis/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/05_bookstore_merge/data/orders_2025_03.csv',
    'stores.csv': 'https://raw.githubusercontent.com/roberthsu2003/PythonForDataAnalysis/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/05_bookstore_merge/data/stores.csv',
    'products.csv': 'https://raw.githubusercontent.com/roberthsu2003/PythonForDataAnalysis/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/05_bookstore_merge/data/products.csv',
}
for name, url in files.items():
    if not (data_dir / name).exists():
        urlretrieve(url, data_dir / name)
print('📂 data/ 內的檔案：', sorted(p.name for p in data_dir.iterdir()))

# 2. 安裝與設定 Colab 繁體中文字型 (思源黑體)
%pip install -q mplfonts
import glob
import mplfonts
from matplotlib import font_manager
from mplfonts import use_font

for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
    font_manager.fontManager.addfont(f)
use_font("Noto Sans CJK SC")

print("✅ 環境初始化完成！資料已就緒，繁體中文字型已設定成功。請開始進行下方單元。")

### 📝 單元一【必做】：讀取五個檔案，比對欄名與鑰匙寫法（貼標籤）

> 🎯 **思維重點**：合併前先「對照」：三個月的欄名一樣嗎？門市代碼有幾種寫法？主檔裡有哪些門市與商品？

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請幫我用 pandas 讀取 data/ 中的 5 個檔案：orders_2025_01.csv、orders_2025_02.csv、orders_2025_03.csv、stores.csv、products.csv。
1. 每個檔案顯示列數與欄位名稱，並說明「一列代表什麼」。
2. 比較三個月訂單的欄位名稱是否完全相同。
3. 列出二月訂單「門市代碼」的所有不同寫法。
4. 列出門市主檔與商品主檔的全部內容。
5. 用文字回報：哪些欄位是合併用的鑰匙（標識）、哪些是條件、哪些是指標？
先不要合併，也不要畫圖。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
import pandas as pd

monthly = {m: pd.read_csv(f'data/orders_2025_{m}.csv') for m in ['01', '02', '03']}
stores = pd.read_csv('data/stores.csv')
products = pd.read_csv('data/products.csv')

for m, table in monthly.items():
    print(f"{m} 月訂單：{len(table)} 列，欄位 {list(table.columns)}")
print("\n二月的門市代碼寫法：", sorted(monthly['02']['門市代碼'].unique()))

print("\n--- 門市主檔（一列 = 一家門市）---")
display(stores)
print("--- 商品主檔（一列 = 一種書）---")
display(products)

### 📝 單元二【必做】：邊界檢核：上下合併、統一鑰匙、左右對應並清點未匹配

> 🎯 **思維重點**：先示範「直接合併」會出什麼問題，再依序修正：統一欄名 → 上下合併 → 統一門市代碼 → 用 left join 對應主檔 → 清點對不上的資料。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請幫我把三個月訂單整理成一張完整的季報資料：
1. 先示範：不改欄名就直接上下合併，會多出哪些欄位、各有多少缺值？（只示範，不要用這個版本繼續）
2. 把三月的「購買數量」改名為「數量」，再上下合併三個月訂單，印出合併前各月筆數與合併後總筆數。
3. 門市代碼去除前後空白並轉成大寫，印出統一前後的不同寫法數量。
4. 用 left join 依序對應商品主檔（取得書名、類別、售價）與門市主檔（取得門市、縣市），印出合併後筆數，並用 indicator 統計各有幾筆對不上；列出對不上的門市代碼與商品編號。
5. 計算營收＝數量 × 售價；沒有售價的訂單營收保持缺值，不要猜價格。再列出「主檔中有、但沒有任何訂單」的門市。
不要修改原始檔案。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
# 示範：欄名不一致就直接上下合併
naive = pd.concat(monthly.values(), ignore_index=True)
print("❌ 直接合併的欄位：", list(naive.columns))
print("   缺值數：", naive[['數量', '購買數量']].isna().sum().to_dict())

# ① 統一欄名後上下合併
orders = pd.concat([t.rename(columns={'購買數量': '數量'}) for t in monthly.values()], ignore_index=True)
print(f"\n① 各月筆數 {[len(t) for t in monthly.values()]} → 合併後 {len(orders)} 筆")

# ② 統一門市代碼寫法
before = orders['門市代碼'].nunique()
orders['門市代碼'] = orders['門市代碼'].str.strip().str.upper()
print(f"② 門市代碼寫法：{before} 種 → {orders['門市代碼'].nunique()} 種 {sorted(orders['門市代碼'].unique())}")

# ③ left join 對應主檔，保留所有訂單
sales = (orders.merge(products, on='商品編號', how='left', indicator='商品比對')
               .merge(stores, on='門市代碼', how='left', indicator='門市比對'))
print(f"③ 合併後筆數：{len(sales)}（應與 {len(orders)} 相同）")
no_product = sales[sales['商品比對'] == 'left_only']
no_store = sales[sales['門市比對'] == 'left_only']
print(f"   對不到商品：{len(no_product)} 筆，商品編號 {sorted(no_product['商品編號'].unique())}，共 {no_product['數量'].sum()} 本")
print(f"   對不到門市：{len(no_store)} 筆，門市代碼 {sorted(no_store['門市代碼'].unique())}")

# ④ 營收：沒有售價就保持缺值，不猜價格
sales['營收'] = sales['數量'] * sales['售價']
sales['門市'] = sales['門市'].fillna(sales['門市代碼'] + '（門市資料缺）')
print("④ 主檔有、但沒有任何訂單的門市：", sorted(set(stores['門市代碼']) - set(orders['門市代碼'])))

inner_revenue = orders.merge(products, on='商品編號').merge(stores, on='門市代碼').eval('數量 * 售價').sum()
print(f"\n💡 left join 可計價營收 {sales['營收'].sum():,.0f} 元；若用 inner join 只剩 {inner_revenue:,.0f} 元，而且程式不會報錯。")

### 📝 單元三【必做】：提問積木【比較】— 第一季哪家門市營收最高？（垂直長條圖）

> 🎯 **思維重點**：6 個門市比高低，用【垂直長條圖】；標題註明「不含無法計價的訂單」，資料不完整的 S07 用灰色區分。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請依門市加總第一季可計價營收，並畫出垂直長條圖：
1. 先產出表格：門市、可計價營收、訂單筆數、無法計價筆數，依營收由高到低排序。
2. X 軸為門市，Y 軸為營收（元），從 0 開始；柱子上方標註金額（千分位）。
3. 營收最高的門市用醒目顏色；門市資料缺的 S07 用灰色。
4. 標題註明「不含無法計價的 P013 訂單」，圖表文字全部使用繁體中文。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
import matplotlib.pyplot as plt

store_df = sales.groupby('門市').agg(可計價營收=('營收', 'sum'), 訂單筆數=('訂單編號', 'count'),
                                    無法計價筆數=('售價', lambda s: s.isna().sum()))
store_df = store_df.sort_values('可計價營收', ascending=False)
display(store_df)

top_store = store_df['可計價營收'].idxmax()
colors = ['#95a5a6' if '資料缺' in s else '#e74c3c' if s == top_store else '#3498db' for s in store_df.index]
plt.figure(figsize=(9, 5))
bars = plt.bar(store_df.index, store_df['可計價營收'], color=colors, width=0.6)
for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, h + 2000, f'{h:,.0f}', ha='center', va='bottom', fontsize=10)
plt.ylim(0, store_df['可計價營收'].max() * 1.15)
plt.title(f'第一季各門市營收（最高：{top_store}；不含無法計價的 P013 訂單）', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('門市')
plt.ylabel('可計價營收（元）')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

## ✅ 必做主線驗收：現在先停下來

✏️ **先不要看答案！** 對照上面的表格和圖，自己回答下列問題，把你的答案寫在紙上或筆記裡：

- [ ] 能說出訂單、門市、商品三種表「一列代表什麼」，以及用哪個欄位把它們接起來。
- [ ] 能展示合併前後筆數一致，並列出對不上的門市（S07）與商品（P013）各影響幾筆。
- [ ] 能用一句話解釋：為什麼這一例用 left join，而不是 inner join？
- 💬 **討論題**：如果改用 inner join，總營收會怎樣？為什麼程式不會報錯？

<details>
<summary>👉 寫好了嗎？點此展開參考答案核對</summary>

- 三個月訂單接起來共 **1,130 筆**，只有一個「數量」欄；門市代碼統一後只剩 S01～S05、S07 六種。
- 合併後仍是 1,130 筆：S07 有 82 筆對不到門市、P013 有 190 筆對不到售價（301 本無法計價）。
- 長條圖與表格都顯示**台北車站店**第一季營收最高（170,009 元）；S07 以灰色標示「門市資料缺」。

</details>

若今天繼續做報表，請進行**單元四**；單元五、六是選做挑戰。

---

### 📝 單元四【報表應用】：匯出門市營收與未匹配清單

> 🎯 **思維重點**：季報要誠實交代缺口：除了門市營收，也附上「未匹配清單」，讓主管知道哪些資料需要補登。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請將季報結果匯出成 Excel：
1. 存為 output/第一季門市營收.xlsx（若 output 目錄不存在請自動建立）。
2. 工作表「門市營收」：上一步的門市營收表。
3. 工作表「未匹配清單」：問題類型（訂單缺門市主檔／訂單缺商品主檔／主檔門市無訂單）、代碼、影響訂單筆數、影響本數。
4. 匯出後重新讀取，核對門市營收合計與合併資料的可計價營收一致。
5. 提供 Colab 下載該 Excel 檔案的程式碼。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
issues = pd.DataFrame([
    ('訂單缺門市主檔', code, len(g), g['數量'].sum()) for code, g in no_store.groupby('門市代碼')
] + [
    ('訂單缺商品主檔', code, len(g), g['數量'].sum()) for code, g in no_product.groupby('商品編號')
] + [
    ('主檔門市無訂單', code, 0, 0) for code in sorted(set(stores['門市代碼']) - set(orders['門市代碼']))
], columns=['問題類型', '代碼', '影響訂單筆數', '影響本數'])
display(issues)

os.makedirs('output', exist_ok=True)
output_path = 'output/第一季門市營收.xlsx'
with pd.ExcelWriter(output_path) as writer:
    store_df.reset_index().to_excel(writer, sheet_name='門市營收', index=False)
    issues.to_excel(writer, sheet_name='未匹配清單', index=False)
check = pd.read_excel(output_path, sheet_name='門市營收')
print(f"重新讀取營收合計：{check['可計價營收'].sum():,.0f}；合併資料可計價營收：{sales['營收'].sum():,.0f}")

try:
    from google.colab import files
    files.download(output_path)
    print("✅ 已啟動瀏覽器下載 Excel 報表！")
except Exception:
    print(f"✅ Excel 檔案已儲存於：{output_path}")

### 📝 單元五【選做挑戰】：提問積木【比較＋時間】— 各門市三個月的營收變化（分組長條圖）

> 🎯 **思維重點**：「類別 ＋ 時間 ＋ 指標」看各群體走勢是否一致；只有 3 個月，用【分組長條圖】比折線更清楚，也不宜稱為「趨勢」。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請比較各門市 1、2、3 月的可計價營收：
1. 從日期取出月份，依門市與月份加總營收，整理成「列＝門市、欄＝月份」的表格。
2. 畫出分組長條圖：X 軸為門市，每個門市 3 根柱子代表 1、2、3 月，Y 軸從 0 開始，加上圖例。
3. 用文字描述哪些門市的月營收變化方向不同，並提醒只有 3 個月，不能稱為長期趨勢。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
sales['月份'] = pd.to_datetime(sales['日期']).dt.month.astype(str) + '月'
month_table = sales.pivot_table(index='門市', columns='月份', values='營收', aggfunc='sum')
month_table = month_table.loc[store_df.index]
display(month_table)

ax = month_table.plot.bar(figsize=(10, 5), color=['#a9cce3', '#5499c7', '#1f618d'], width=0.75, rot=0)
ax.set_ylim(0, month_table.max().max() * 1.15)
ax.set_title('各門市 1～3 月可計價營收', fontsize=13, fontweight='bold')
ax.set_xlabel('門市')
ax.set_ylabel('可計價營收（元）')
ax.legend(title='月份')
plt.show()
print("⚠️ 邊界提醒：只有 3 個月，只能描述這一季的變化，不能稱為長期趨勢。")

### 📝 單元六【選做挑戰】：提問積木【組成】— 各門市的書籍類別組成（百分比堆疊長條圖）

> 🎯 **思維重點**：「組成」要先寫清楚分母：這裡的分母是**每家門市自己的可計價營收**，所以每根長條都是 100%。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請分析各門市可計價營收的書籍類別組成：
1. 只使用有售價的訂單，依門市與類別加總營收，再除以該門市的可計價營收，換算成百分比。
2. 畫出水平的百分比堆疊長條圖：每家門市一根長條（總和 100%），不同顏色代表類別，加上圖例。
3. 在圖表標題或註解寫明「分母＝該門市可計價營收」。
4. 用文字指出哪一個類別在多數門市占比最高。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
priced = sales.dropna(subset=['售價'])
mix = priced.pivot_table(index='門市', columns='類別', values='營收', aggfunc='sum', fill_value=0)
mix_pct = mix.div(mix.sum(axis=1), axis=0) * 100
display(mix_pct.round(1))

ax = mix_pct.plot.barh(stacked=True, figsize=(10, 5), colormap='Set2', width=0.7)
ax.set_xlim(0, 100)
ax.set_xlabel('占該門市可計價營收（%）')
ax.set_ylabel('門市')
ax.set_title('各門市營收的類別組成（分母＝該門市可計價營收）', fontsize=13, fontweight='bold')
ax.legend(title='類別', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()
print('各門市占比最高的類別：', mix_pct.idxmax(axis=1).to_dict())

---

## 🎯 學習驗收與成果回顧

完成目前選擇的層次後，請檢視你是否能親自驗收：
- [ ] **必做**：三個月合併 1,130 筆、門市代碼統一、left join 後筆數不變；台北車站店營收最高。
- [ ] **報表應用**：Excel 含門市營收與未匹配清單，營收合計與合併資料一致。
- [ ] **選做挑戰**：分組長條圖只描述 3 個月的變化；百分比堆疊長條圖寫明分母。
- [ ] **AI 協作**：能說明自己決定的合併方式、AI 幫忙執行的事，以及自己如何確認結果。